# [센서 실습 21] 소음 센서 – MAX4466 마이크 증폭 모듈

마이크 증폭 모듈 **MAX4466**의 음성 파형을 ADC로 빠르게 읽어 **최고-최저 전압차(Vpp)** 를 구하고, 소음계로 맞춘 **보정값**으로 dB를 계산합니다. 1분 단위 **등가소음도(Leq)** 도 구합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
마이크는 공기의 떨림(소리)을 **전압의 떨림**으로 바꿉니다. MAX4466은 이 작은 신호를 증폭해 **1.65V를 중심으로 위아래로 흔들리는 파형**을 내보냅니다.
- 소리가 클수록 파형의 흔들림 폭(**Vpp**)이 커짐
- 소리의 세기는 로그 단위인 **데시벨(dB)** 로 표현: 소리 에너지가 10배 → +10dB, 전압 진폭이 10배 → +20dB

$$L = L_{ref} + 20 \log_{10}\left(\frac{V_{pp}}{V_{pp,ref}}\right)$$

- $L_{ref}, V_{pp,ref}$: **소음계(또는 스마트폰 소음 측정 앱)** 를 옆에 두고 같은 소리를 잰 보정값
- 저가 마이크 모듈은 주파수 특성이 고르지 않아 **정확한 법정 측정용은 아님** → 상대적 변화 감시용

| 소리 | 대략적인 크기 |
|---|---|
| 조용한 도서관 | 35 ~ 40 dB |
| 일반 대화 | 55 ~ 65 dB |
| 시끄러운 교실·식당 | 70 ~ 80 dB |
| 공사장 굴착기 | 85 ~ 100 dB |

### 부품 사양
| 항목 | 내용 |
|---|---|
| 마이크 | 일렉트릿 콘덴서 마이크 |
| 증폭 | 25 ~ 125배 (가변저항 조절) |
| 출력 | 아날로그 (VCC/2 중심 교류) |
| 주파수 특성 | 20Hz ~ 20kHz |
| 전원 | 2.4 ~ 5.5V (3.3V 사용) |

## 2. 라즈베리파이 연결 방법
**준비물**: MAX4466 모듈, MCP3008, 점퍼선 (+ 보정용 소음계 또는 스마트폰 소음 측정 앱)

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| MAX4466 VCC / GND | 3.3V / GND | 3.3V로 공급하면 출력도 0~3.3V |
| MAX4466 OUT | MCP3008 CH0 (1번) |  |



**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
짧은 시간 동안 모은 파형에서 Vpp를 구하고, **보정값**을 이용해 dB로 바꿉니다.

In [ ]:
VREF = 3.3
REF_DB, REF_VPP = 40.0, 0.012      # 보정: 소음계 40dB일 때 측정한 Vpp가 0.012V

def samples_to_vpp(adc_values):
    v = np.asarray(adc_values) * VREF
    return float(v.max() - v.min())

def vpp_to_db(vpp):
    return round(REF_DB + 20 * np.log10(max(vpp, 1e-4) / REF_VPP), 1)

for vpp in (0.012, 0.038, 0.12, 0.38):
    print(f"Vpp {vpp:.3f}V → {vpp_to_db(vpp)} dB")

### 4-2. 실제 센서 클래스
한 번 `read()`할 때 ADC를 **200회 연속**으로 읽어 파형을 모은 뒤 Vpp와 dB를 계산합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMAX4466:
    def __init__(self, channel=0, n=200):
        from gpiozero import MCP3008
        self.adc, self.n = MCP3008(channel=channel), n

    def read(self):
        samples = [self.adc.value for _ in range(self.n)]      # 짧은 시간 동안 파형 수집
        vpp = samples_to_vpp(samples)
        return {"vpp": round(vpp, 4), "db": vpp_to_db(vpp)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 시간 | 가정한 상황 | 소리 크기 |
|---|---|---|
| 0 ~ 2분 | 조용한 사무실 | 약 45 dB |
| 2 ~ 4분 | 회의(대화) | 약 60 dB |
| 4 ~ 7분 | 창밖 공사 소음 | 약 78 dB |
| 5분 10초 | 문이 쾅 닫힘 | 순간 90 dB |
| 7 ~ 10분 | 다시 조용 | 약 45 dB |

가상 센서는 dB로부터 진폭을 계산해 **200개짜리 음성 파형(ADC 값)** 을 만들고, 실제와 같은 Vpp 계산과 보정식을 거칩니다.

In [ ]:
class VirtualMAX4466:
    def __init__(self, dt=1.0, seed=21, n=200, fs=8000):
        self.dt, self.t, self.n, self.fs = dt, 0.0, n, fs
        self.rng = np.random.default_rng(seed)

    def _true_db(self):
        m = self.t / 60
        db = 45 if m < 2 or m >= 7 else 60 if m < 4 else 78
        if abs(self.t - 310) < 1: db = 90                       # 문 닫히는 소리
        return db + self.rng.normal(0, 1.5)

    def read(self):
        vpp = REF_VPP * 10 ** ((self._true_db() - REF_DB) / 20)
        tt = np.arange(self.n) / self.fs
        f = self.rng.uniform(200, 1200)
        wave = 0.5 * vpp * (0.7 * np.sin(2 * np.pi * f * tt) + 0.3 * np.sin(2 * np.pi * 2.3 * f * tt))
        wave *= vpp / max(1e-9, wave.max() - wave.min())        # 파형 높이를 vpp에 맞춤
        samples = np.clip((1.65 + wave) / VREF, 0, 1)           # ADC 값 0~1 (1.65V 중심)
        self.t += self.dt
        vpp_m = samples_to_vpp(samples)
        return {"vpp": round(vpp_m, 4), "db": vpp_to_db(vpp_m)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMAX4466(dt=1.0)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1초마다 소리 크기를 측정해 10분 동안 기록합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 600               # 측정 횟수
INTERVAL = 1.0               # 측정 간격(초)

sensor = RealMAX4466() if USE_REAL_SENSOR else VirtualMAX4466(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. **1분 등가소음도(Leq)**: 변하는 소음을 같은 에너지의 일정한 소음으로 바꾼 값 $L_{eq} = 10\log_{10}\left(\frac{1}{N}\sum 10^{L_i/10}\right)$
2. Leq가 **70dB 이상**이면 소음 경보(교육용 기준), 순간 최대값도 기록합니다.

In [ ]:
df["minute"] = (df["time_s"] // 60).astype(int)
leq = df.groupby("minute")["db"].apply(lambda x: 10 * np.log10(np.mean(10 ** (x / 10)))).round(1)
lmax = df.groupby("minute")["db"].max()
summary = pd.DataFrame({"Leq(dB)": leq, "최대(dB)": lmax})
summary["판정"] = np.where(summary["Leq(dB)"] >= 70, "소음 경보", "정상")
print(summary.to_string())
print(f"\n10분 전체 Leq: {10 * np.log10(np.mean(10 ** (df.db / 10))):.1f} dB "
      f"(산술 평균 {df.db.mean():.1f} dB와 비교 – 큰 소리의 영향이 더 크게 반영됨)")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df.time_s / 60, df.db, color="gray", lw=0.8, label="1초 측정값")
ax.step(summary.index, summary["Leq(dB)"], "k-", where="post", lw=2, label="1분 Leq")
ax.axhline(70, color="gray", ls="--", label="경보 70 dB")
ax.set_title("MAX4466 소음 측정"); ax.set_xlabel("시간 (분)"); ax.set_ylabel("dB")
ax.grid(alpha=0.3); ax.legend(loc="upper right"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "소음센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MAX4466 마이크 증폭 모듈",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/noise", payload)
# client.disconnect()

## 직접 해 보기
- 보정값 `REF_VPP`를 0.02로 바꾸면 모든 dB가 얼마나 달라지나요? (보정의 중요성)
- Leq와 산술 평균이 왜 다른지 설명해 보세요.
- 밤 10시 이후에는 기준을 55dB로 낮추는 시간대별 판정을 만들어 보세요.